<h1>Decision Tree Model</h1>

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("../data/Loan_default.csv")
df.drop('LoanID',axis=1,inplace=True)
df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


<h3>Categorical String Data into numeric data</h3>

In [4]:
from sklearn.preprocessing import LabelEncoder

In [5]:
leHasMortgage = LabelEncoder()
df['HasMortgage'] =  leHasMortgage.fit_transform(df[['HasMortgage']])

leHasDependents = LabelEncoder()
df['HasDependents'] =  leHasDependents.fit_transform(df[['HasDependents']])

leHasCoSigner = LabelEncoder()
df['HasCoSigner'] =  leHasCoSigner.fit_transform(df[['HasCoSigner']])

In [6]:
from sklearn.preprocessing import OneHotEncoder
columns_to_encode = [
    'Education',
    'EmploymentType',
    'MaritalStatus',
    'LoanPurpose'
]

ohe = OneHotEncoder(sparse_output=False)

encoded = ohe.fit_transform(df[columns_to_encode])

encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(columns_to_encode),
    index=df.index
)

df = pd.concat(
    [
        df.drop(columns=columns_to_encode),
        encoded_df
    ],
    axis=1
)

<h3>Spliting Data</h3>

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop('Default',axis=1)
y = df['Default']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

<h3>Model Training<h3>

In [8]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=8)

model.fit(X_train,y_train)

print(f"Accuracy in Training = : {round(model.score(X_train,y_train),2)}")
print(f"Accuracy in Testing = : {round(model.score(X_test,y_test),2)}")

Accuracy in Training = : 0.89
Accuracy in Testing = : 0.88


<h3>1. Confusion Matrix</h3>

In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,model.predict(X_test))
tn, fp, fn, tp = cm.ravel()
print(cm)
#                  Predicted
#                  0       1
# Actual  0       TN      FP
#         1       FN      TP
# 0 : Non Default
# 1 : Default

[[44941   198]
 [ 5695   236]]


<h3>2. Accuracy (Recognition Rate)</h3>

In [10]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test,model.predict(X_test))
# same as model.score(X_test,y_test)
print(f"Accuracy in testing : {round(acc,2)}")

Accuracy in testing : 0.88


In [11]:
# Formula: (TP + TN) / Total — "what fraction did I get right overall?"
accUsingFormuala = (tp+tn)/len(y_test)
print(f"Accuracy in testing : {round(accUsingFormuala,2)}")

Accuracy in testing : 0.88


<h3>3. Error Rate (Misclassification Rate)</h3>

In [12]:
erroUsingFormula = (fn+fp)/len(y_test)
print(f"Error in testing : {round(erroUsingFormula,2)}")

# 1−accuracy(M)

Error in testing : 0.12


<h3>4. Precision and Recall</h3>

In [13]:
precision = tp/(tp+fp)
print(f"Of everyone I flagged as a defaulter, {round(precision,2)} actually defaulted")

Of everyone I flagged as a defaulter, 0.54 actually defaulted


In [14]:
recall = tp/(fn+tp)
print(f"Of everyone who actually defaulted, {round(recall,2)} did I catch")

Of everyone who actually defaulted, 0.04 did I catch


In [15]:
from sklearn.metrics import precision_score,recall_score
print(f"Precision score = {round(precision_score(y_test,model.predict(X_test)),2)}")
print(f"Recall score = {round(recall_score(y_test,model.predict(X_test)),2)}")

Precision score = 0.54
Recall score = 0.04


<h3>5. F1 and F-beta Score</h3>

In [16]:
from sklearn.metrics import f1_score,fbeta_score

# balanced assessment of a
# model's performance by taking both false
# positives and false negatives into account.
print(f"F1 score : {f1_score(y_test,model.predict(X_test))}")

# β > 1, it emphasizes recall more than
# precision,
print(f"Fbeta score (recall) : {fbeta_score(y_test,model.predict(X_test),beta=2)}")

# when 0 < β < 1, it emphasizes
# precision more than recall.
print(f"Fbeta score (precision) : {fbeta_score(y_test,model.predict(X_test),beta=0.5)}")

F1 score : 0.07415553809897879
Fbeta score (recall) : 0.048845103071446315
Fbeta score (precision) : 0.15390635189774357


<h3>Classification report</h3>

In [17]:
from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.54      0.04      0.07      5931

    accuracy                           0.88     51070
   macro avg       0.72      0.52      0.51     51070
weighted avg       0.85      0.88      0.84     51070



<h2>Cross-Validation (K-Fold)</h2>

Cross-Validation
        ↓
How well does my model generalize?

In [18]:
from sklearn.model_selection import cross_val_score,StratifiedKFold

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state = 42
)


#Train only on training data so that we have the testing set untouched
# so that later on we can test on the test data for accuracy

modelUsingKFold = DecisionTreeClassifier()

scores = cross_val_score(
    modelUsingKFold,
    X_train,
    y_train,
    cv=skf,
    scoring='accuracy'
)

# CV accuracy = correct predictions on that fold's validation part ÷ total records in that validation part.
# its of testing for each fold
print(f"Scores : {scores}")
print("CV Score: ", scores.mean())

# CV training is for evaluating the model across different splits.
# Final fit() is for creating the model we will actually use.

modelUsingKFold.fit(X_train, y_train)

print(f"Accuracy in Training = : {round(modelUsingKFold.score(X_train,y_train),2)}")

print(f"Accuracy in Testing = : {round(modelUsingKFold.score(X_test,y_test),2)}")

Scores : [0.8025749  0.79875661 0.80639319 0.80140004 0.8015469  0.80218328
 0.80223223 0.7981593  0.79688647 0.80457238]
CV Score:  0.8014705287749777
Accuracy in Training = : 1.0
Accuracy in Testing = : 0.8


<h2>Ada Boost</h2>

Let's train several weak models one after another, and each new model will pay more attention to the examples that the previous model got wrong."

In [19]:
from sklearn.ensemble import AdaBoostClassifier

modelAdaBoost = AdaBoostClassifier(random_state=42,n_estimators=100,learning_rate=1.5)

modelAdaBoost.fit(X_train,y_train)


AdaBoostClassifier(learning_rate=1.5, n_estimators=100, random_state=42)

In [20]:
print(f"Accuracy in Training = : {round(modelAdaBoost.score(X_train,y_train),2)}")

print(f"Accuracy in Testing = : {round(modelAdaBoost.score(X_test,y_test),2)}")

Accuracy in Training = : 0.89
Accuracy in Testing = : 0.89


<h2>Using GridSearchCV</h2>

In [21]:
parm_grid = {
    "n_estimators" : [50,100,200],
    "learning_rate": [0.5, 1.0, 1.5]
}

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),
    param_grid=parm_grid,
    cv=5,
    # scoring="accuracy"
    scoring="recall"   # <-- this is the only real change: optimize for catching defaulters

)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=AdaBoostClassifier(random_state=42),
             param_grid={'learning_rate': [0.5, 1.0, 1.5],
                         'n_estimators': [50, 100, 200]},
             scoring='recall')

best_params_ tells you which combination won.

best_score_ tells you its average CV accuracy.

In [22]:
print(f"Best Parameters : {grid_search.best_params_}")
print(f"Best Score(avg) : {grid_search.best_score_}")

Best Parameters : {'learning_rate': 1.5, 'n_estimators': 50}
Best Score(avg) : 0.06475006086108212


now train model using best params

In [26]:
modelAdaBoost = AdaBoostClassifier(random_state=42,n_estimators=50,learning_rate=1.5)

modelAdaBoost.fit(X_train,y_train)

AdaBoostClassifier(learning_rate=1.5, random_state=42)

In [27]:
print(f"Accuracy in Training = : {round(modelAdaBoost.score(X_train,y_train),2)}")

print(f"Accuracy in Testing = : {round(modelAdaBoost.score(X_test,y_test),2)}")

Accuracy in Training = : 0.89
Accuracy in Testing = : 0.89


In [ ]:
# from sklearn.metrics import fbeta_score

# for threshold in [0.40, 0.42, 0.44, 0.45, 0.46, 0.48, 0.50]:
#     predictions = (probs > threshold).astype(int)
#     f2 = fbeta_score(y_test, predictions, beta=2)
#     print(f"Threshold {threshold}: "
#           f"Accuracy={round(accuracy_score(y_test, predictions),2)}, "
#           f"Recall={round(recall_score(y_test, predictions),2)}, "
#           f"Precision={round(precision_score(y_test, predictions),2)}, "
#           f"F2={round(f2,2)}")

Threshold 0.4: Accuracy=0.39, Recall=0.91, Precision=0.15, F2=0.45
Threshold 0.42: Accuracy=0.59, Recall=0.78, Precision=0.19, F2=0.48
Threshold 0.44: Accuracy=0.75, Recall=0.59, Precision=0.25, F2=0.46
Threshold 0.45: Accuracy=0.81, Recall=0.47, Precision=0.29, F2=0.42
Threshold 0.46: Accuracy=0.84, Recall=0.35, Precision=0.34, F2=0.35
Threshold 0.48: Accuracy=0.88, Recall=0.16, Precision=0.44, F2=0.18
Threshold 0.5: Accuracy=0.89, Recall=0.05, Precision=0.57, F2=0.06


In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelRF = RandomForestClassifier(random_state=42, n_estimators=100)
modelRF.fit(X_train, y_train)

print(f"Accuracy in Training = : {round(modelRF.score(X_train,y_train),2)}")
print(f"Accuracy in Testing = : {round(modelRF.score(X_test,y_test),2)}")

from sklearn.metrics import recall_score, precision_score
preds_default = modelRF.predict(X_test)
print(f"Recall    : {round(recall_score(y_test, preds_default), 2)}")
print(f"Precision : {round(precision_score(y_test, preds_default), 2)}")

Accuracy in Training = : 1.0
Accuracy in Testing = : 0.89
Recall    : 0.03
Precision : 0.64


<h2>Saving the model</h2>

In [25]:
import joblib

# Save the model
joblib.dump(model, "DecisionTreeModel.pkl")

['DecisionTreeModel.pkl']